# 5. Pipeline de regresión lineal

Se construye un modelo de regresión lineal para estimar el salario mensual en quetzales utilizando seis predictores: edad, antigüedad, horas semanales, nivel educativo, categoría ocupacional y dominio.

Se utilizan los primeros tres trimestres de 2025 para entrenar y el cuarto trimestre para validar y seleccionar la regularización. El primer trimestre de 2026 se reserva para la evaluación final.

Como referencia, se utiliza un modelo que predice para todos los registros el salario promedio del conjunto de entrenamiento.

In [7]:
import os
import json
import pandas as pd

from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab7_RegresionLineal")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)

assert spark.version.startswith("3.5."), "El laboratorio requiere Spark 3.5.x."

Versión de Spark: 3.5.1


26/09/25 01:54:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [8]:
PARQUET_DIR = "../working_dir/parquet"
RUTA_2025 = f"{PARQUET_DIR}/eneic_2025_preparado"

NUMERICAS = [
    "edad",
    "antiguedad",
    "horas_semanales"
]

CATEGORICAS = [
    "nivel_educativo",
    "categoria_ocupacional",
    "dominio"
]

OBJETIVO = "salario_mensual"

COLUMNAS_MODELO = [
    "periodo_archivo",
    "NUM_HOGAR",
    "NUM_PERSONA",
    OBJETIVO,
    *NUMERICAS,
    *CATEGORICAS
]

df_2025_modelo = (
    spark.read.parquet(RUTA_2025)
    .select(*COLUMNAS_MODELO)
)

print(f"Registros preparados de 2025: {df_2025_modelo.count():,}")

df_2025_modelo.printSchema()
df_2025_modelo.show(5, truncate=False)

Registros preparados de 2025: 53,025
root
 |-- periodo_archivo: string (nullable = true)
 |-- NUM_HOGAR: long (nullable = true)
 |-- NUM_PERSONA: integer (nullable = true)
 |-- salario_mensual: double (nullable = true)
 |-- edad: double (nullable = true)
 |-- antiguedad: double (nullable = true)
 |-- horas_semanales: double (nullable = true)
 |-- nivel_educativo: string (nullable = true)
 |-- categoria_ocupacional: string (nullable = true)
 |-- dominio: string (nullable = true)

+---------------+---------+-----------+---------------+----+-------------------+---------------+---------------+---------------------+-------+
|periodo_archivo|NUM_HOGAR|NUM_PERSONA|salario_mensual|edad|antiguedad         |horas_semanales|nivel_educativo|categoria_ocupacional|dominio|
+---------------+---------+-----------+---------------+----+-------------------+---------------+---------------+---------------------+-------+
|2025T1         |17085    |1          |5000.0         |33.0|8.0                |40.0   

## Entrenamiento y validación

In [9]:
train = (
    df_2025_modelo
    .filter(F.col("periodo_archivo").isin("2025T1", "2025T2", "2025T3"))
    .cache()
)

validacion = (
    df_2025_modelo
    .filter(F.col("periodo_archivo") == "2025T4")
    .cache()
)

n_train = train.count()
n_validacion = validacion.count()

assert n_train > 0, "El conjunto de entrenamiento está vacío."
assert n_validacion > 0, "El conjunto de validación está vacío."

print(f"Entrenamiento: {n_train:,} registros")
print(f"Validación:    {n_validacion:,} registros")

(
    train.withColumn("conjunto", F.lit("Entrenamiento"))
    .unionByName(
        validacion.withColumn("conjunto", F.lit("Validación"))
    )
    .groupBy("conjunto", "periodo_archivo")
    .count()
    .orderBy("periodo_archivo")
    .show(truncate=False)
)

Entrenamiento: 40,361 registros
Validación:    12,664 registros
+-------------+---------------+-----+
|conjunto     |periodo_archivo|count|
+-------------+---------------+-----+
|Entrenamiento|2025T1         |13419|
|Entrenamiento|2025T2         |13492|
|Entrenamiento|2025T3         |13450|
|Validación   |2025T4         |12664|
+-------------+---------------+-----+



## Preparación del pipeline y criterios de evaluación

Las variables categóricas se procesan con StringIndexer y OneHotEncoder. Esta codificación evita interpretar sus códigos como cantidades o como una escala numérica.

StringIndexer utiliza handleInvalid="keep" para conservar registros con categorías no observadas durante el entrenamiento.

VectorAssembler combina los tres predictores numéricos y las tres variables categóricas codificadas. Se utiliza la estandarización interna de LinearRegression mediante standardization=True.

Todos los componentes del pipeline se ajustan únicamente con entrenamiento.

Se calculan tres métricas sobre todos los registros de validación:

- MAE
- RMSE
- R²

La configuración se selecciona por el menor RMSE de validación. Además, se compara con una referencia que utiliza exclusivamente la media salarial del entrenamiento.